# Intelli-Speech — SYNTH_SOURCE_V1 (Phase 3 source speech: pilot and full)

Typical Telugu speech from regular Vaani that will later be turned into synthetic atypical speech.

- **Excluded speakers:** everyone already used as a control (`typical_controls_v2`, `vaani_telugu_transcribed`).
- **Selection:** at most 20 clips per speaker, 2–30 s, gender-balanced; transcripts not required.
- **Quality screen:** SNR estimate, bandwidth, clipping, amount of speech. Rejected clips are only marked, never deleted.
- **Split:** train/val/test by speaker, 80/10/10.

**Set `SET` in cell 1.**
- **Pilot (done):** 375 usable clips from 87 speakers, in `synth_source_v1_pilot_r2/`. Run 1 (`synth_source_v1_pilot/`) failed at selection; ignore it.
- **Full:** 240 shards spread across the dataset; at least 3 clips per speaker; up to 7,000 clips selected, of which about 4,000 should be usable (about 1,000 speakers, about 5 h). The output goes to `synth_source_v1_full/`.

The download needs up to 25 GB of network transfer but writes only about 1 GB to Drive.

**Do not commit this notebook with outputs.** Speaker IDs are never printed here.

In [ ]:
# CELL 1 | LOGIN + SYNC + DEPENDENCIES
from huggingface_hub import notebook_login, whoami
try:
    print("Logged in as:", whoami()["name"])
except Exception:
    notebook_login()
import os, subprocess, sys, json, shutil
import pandas as pd
from pathlib import Path
REPO_DIR = Path("/content/Intelli-Speech")
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "https://github.com/niharika9nx/Intelli-Speech.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))
assert Path("scripts/build_synth_source.py").is_file(), "Pull the latest code first."
!pip install -q datasets soundfile praat-parselmouth pyarrow "librosa==0.10.2.post1" "numpy<2.5"
DATA_ROOT = "/content/drive/MyDrive/Intelli-Speech-data"
SET = "full"   # "pilot" = finished pilot (synth_source_v1_pilot_r2); "full" = the ~5 h source set
OUT = Path(DATA_ROOT) / ("synth_source_v1_pilot_r2" if SET == "pilot" else "synth_source_v1_full")
MIN_SELECTED = 500 if SET == "pilot" else 3000
MIN_PASS = 3 if SET == "pilot" else 2
ARGS = f'--set {SET} --data-root "{DATA_ROOT}" --output-dir "{OUT}"'
print("set:", SET, "| output:", OUT.name)

In [ ]:
# CELL 2 | PLAN (metadata only, no audio; full set scans 240 shards, ~20-40 min)
!python scripts/build_synth_source.py --phase plan {ARGS}
plan = json.loads((OUT / "plan.json").read_text())
for k in ("shards_scanned", "exclusions", "rows_removed_by_exclusion", "invalid_speaker_id_rows", "missing_utterance_id_rows",
          "clips_per_speaker_seen", "speakers_seen", "eligible_speakers",
          "selected_speakers", "selected_rows", "selected_with_transcript", "shortfall", "selected_gender_speakers",
          "selected_districts"):
    print(f"{k:28s} {plan.get(k)}")
print(f"estimated download: {plan['estimated_audio_bytes'] / 1e9:.2f} GB")

In [ ]:
# CELL 3 | DOWNLOAD (only the selected rows; hard size cap + Drive space check)
MAX_TRANSFER_GB = 25.0  # network transfer from HF (whole row groups), not Drive storage
plan = json.loads((OUT / "plan.json").read_text())
est_gb = plan["estimated_audio_bytes"] / 1e9
free_gb = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
wav_gb = pd.read_csv(OUT / "selection_plan.csv")["duration"].sum() * 32000 / 1e9
print(f"estimated transfer {est_gb:.2f} GB; WAVs written to Drive ~{wav_gb:.2f} GB; Drive free ~{free_gb:.1f} GB")
assert plan["selected_rows"] >= MIN_SELECTED, "Selection too small; stop and report to Claude."
print("Full download can take 1-2 h: keep the Colab tab open. If it disconnects, rerun this cell (no manifest is written until the end).")
assert est_gb <= MAX_TRANSFER_GB, "Transfer estimate above cap; stop and report to Claude."
assert free_gb > 2.0, "Less than 2 GB free on Drive; free space first."
!python scripts/build_synth_source.py --phase download {ARGS}

In [ ]:
# CELL 4 | QUALITY SCREEN + SPEAKER SPLIT (pilot ~5 min; full ~30-45 min CPU)
!python scripts/build_synth_source.py --phase quality {ARGS} --min-pass-per-speaker {MIN_PASS}

In [ ]:
# CELL 5 | SUMMARY (counts only, no speaker IDs)
import pandas as pd
q = pd.read_csv(OUT / "quality.csv", dtype={"speakerID": str})
print("pass rate:", round(q.quality_pass.mean(), 3), "| usable clips:", int(q.usable.sum()),
      "| usable minutes:", round(q.loc[q.usable, "duration_s"].sum() / 60, 1))
print(); print("reject reasons:"); print(q.loc[~q.quality_pass, "reject_reasons"].str.split(";").explode().value_counts().to_string())
print(); print("by split:")
print(q[q.usable].groupby("split").agg(speakers=("speakerID", "nunique"), clips=("clip_id", "size"),
      minutes=("duration_s", lambda d: round(d.sum() / 60, 1))).to_string())
print(); print("metric quantiles (all screened clips):")
print(q[["snr_db_estimate", "effective_bandwidth_hz", "speech_activity_ratio", "speech_time_s"]].describe().round(2).to_string())

Save the notebook and tell Claude. Do not commit it with outputs.